# Lesson 8: Understanding Reasoning Models

## Weird AI Reasoning Foundations

In the first part of the course, Weird AI moved through the standard LLM project stages:

1. Tokenization
2. Attention
3. Transformer blocks
4. Pretraining
5. Classification fine-tuning
6. Instruction fine-tuning

This lesson begins the second textbook, *Build a Reasoning Model from Scratch*.  The goal of this notebook is not to train a reasoning model yet.  Instead, you will explore what it means for a model to produce intermediate reasoning steps and how that idea connects to the Weird AI parody assistant.

By the end of this notebook, you should be able to:

- Explain what reasoning means in the context of LLMs
- Distinguish pattern matching from reasoning-like behavior
- Explain closed-world and open-world reasoning
- Build simple prompts that encourage step-by-step reasoning
- Decide when reasoning is useful and when it is unnecessary

## 1. Reasoning in LLMs

In this course, we will use **reasoning** in the practical engineering sense used by the textbook:

> A reasoning model produces intermediate steps before producing a final answer.

Those intermediate steps may be visible to the user, hidden inside special tags, or used internally by the system.  The important idea is that the model spends extra tokens working through the problem instead of jumping directly to the final answer.

For Weird AI, this matters because parody generation can involve several smaller decisions:

- What topic is being parodied?
- What original idea or style is being transformed?
- What humorous contrast should be used?
- What rhyme or rhythm pattern should be followed?
- What final lyric should be produced?

In [1]:
# Run this cell to compare a direct prompt with a reasoning-style prompt.

direct_prompt = "Write a silly parody chorus about computer networking."

reasoning_prompt = """
Write a silly parody chorus about computer networking.

Think step-by-step before writing the final chorus:
1. Identify the main topic.
2. List funny networking-related words or situations.
3. Pick a simple rhyme pattern.
4. Write the final chorus.
"""

print("DIRECT PROMPT:")
print(direct_prompt)
print("\n" + "-" * 60 + "\n")
print("REASONING-STYLE PROMPT:")
print(reasoning_prompt)

DIRECT PROMPT:
Write a silly parody chorus about computer networking.

------------------------------------------------------------

REASONING-STYLE PROMPT:

Write a silly parody chorus about computer networking.

Think step-by-step before writing the final chorus:
1. Identify the main topic.
2. List funny networking-related words or situations.
3. Pick a simple rhyme pattern.
4. Write the final chorus.



### Exercise 1

In the markdown cell below, answer these questions:

1. Which prompt gives the model more structure?
2. Which prompt is likely to produce a more deliberate answer?
3. What extra cost might the reasoning-style prompt create?

### Student Response: Exercise 1

TODO: Write your answer here.

## 2. Pattern Matching vs Reasoning

A conventional LLM often appears to reason because it has seen many reasoning-like patterns during training.  However, the model is still generating one token at a time based on learned statistical patterns.

For example, if the prompt is:

```text
The capital of Germany is...
```

A model may answer:

```text
Berlin
```

That does not require logical deduction.  It can be answered through a strong learned association.

A reasoning task is different when the answer depends on multiple premises, intermediate steps, or detecting a contradiction.

In [2]:
pattern_prompt = "The capital of Germany is..."

reasoning_task = """
All birds can fly.
A penguin is a bird.
Can a penguin fly?
"""

print("Pattern-matching style prompt:")
print(pattern_prompt)
print("\nReasoning-style task:")
print(reasoning_task)

Pattern-matching style prompt:
The capital of Germany is...

Reasoning-style task:

All birds can fly.
A penguin is a bird.
Can a penguin fly?



### Exercise 2

Explain why the first prompt can be answered mostly by pattern matching, while the second prompt requires more careful reasoning.

### Student Response: Exercise 2

TODO: Write your answer here.

## 3. Closed-World vs Open-World Reasoning

The textbook uses the example:

```text
All birds can fly.
A penguin is a bird.
Can a penguin fly?
```

There are two ways to evaluate this:

### Closed-world reasoning
Only use the information explicitly given in the prompt.

If all birds can fly, and a penguin is a bird, then the closed-world answer is:

```text
Yes, a penguin can fly.
```

### Open-world reasoning
Use background knowledge in addition to the prompt.

Since we know penguins normally cannot fly, the open-world answer should notice a contradiction between the prompt and real-world knowledge.

In [ ]:
def closed_world_penguin_answer() -> str:
    """
    Return the answer using only the premises in the prompt.
    Do not use background knowledge about real penguins.
    """
    return (
        "Yes, the premises state that all birds can fly and that a penguin is a bird. "
        "Accepting both premises as given, a penguin must be able to fly."
    )


def open_world_penguin_answer() -> str:
    """
    Return the answer using background knowledge.
    The answer should mention the contradiction.
    """
    return (
        "No. In the real world, penguins are birds but they can't fly. "
        "This contradicts the prompt's first premise, 'All birds can fly', "
        "so the prompt contains a false generalisation. Real world knowledge "
        "indicates penguins swim rather than fly."
    )


print("Closed-world answer:", closed_world_penguin_answer())
print("Open-world answer:", open_world_penguin_answer())

Closed-world answer: Yes, the premises state that all birds can fly and that a penguin is a bird. Accepting both premises as given, a penguin must be able to fly.
Open-world answer: No. In the real world, penguins are birds but they can't fly. This contradicts the prompt's first premise, 'All birds can fly', so the prompt contains a false generalisation. Real world knowledge indicates penguins swim rather than fly.


### Exercise 3

In your own words, explain why both answers can be considered reasonable depending on the rules of the task.

### Student Response: Exercise 3

TODO: Write your answer here.

## 4. Reasoning for Weird AI

A parody assistant can benefit from reasoning because parody writing is not just raw text generation.  It often requires planning.

For this lesson, you will not modify the model weights.  Instead, you will create prompt-building helpers that encourage Weird AI to reason through a creative task.

This is an example of **inference-time reasoning support**: changing how we ask the model to respond without retraining the model.

In [9]:
def build_reasoning_prompt(user_prompt: str) -> str:
    """
    Convert a normal user prompt into a reasoning-style prompt.
    """

    return f"""Task:
{user_prompt}

Before producing the final answer, work through the request step-by-step:
1. Restate what the user is asking for in new words.
2. Identify any important constraints, style requirements, or content the response must include based on the prompt.
3. Plan the response; list key points, structure, or ideas you will use.
4. Write the answer clearly and completely.
"""


example = "Write a parody chorus about debugging Python code."
print(build_reasoning_prompt(example))

Task:
Write a parody chorus about debugging Python code.

Before producing the final answer, work through the request step-by-step:
1. Restate what the user is asking for in new words.
2. Identify any important constraints, style requirements, or content the response must include based on the prompt.
3. Plan the response; list key points, structure, or ideas you will use.
4. Write the answer clearly and completely.



In [5]:
# Self-checks for build_reasoning_prompt.
# These are not full pytest tests, but they help you check your work in the notebook.

prompt = build_reasoning_prompt("Write a parody chorus about databases.")

assert isinstance(prompt, str), "The function should return a string."
assert "parody chorus about databases" in prompt.lower(), "The original user request should still be included."
assert "step" in prompt.lower() or "plan" in prompt.lower(), "The prompt should encourage reasoning or planning."
assert "final" in prompt.lower(), "The prompt should ask for a final answer."

print("build_reasoning_prompt self-checks passed.")

build_reasoning_prompt self-checks passed.


## 5. Building a Parody Reasoning Prompt

Now create a more specific helper for Weird AI.

The goal is to produce a prompt that guides the model through parody planning before generating lyrics.

In [17]:
def build_parody_reasoning_prompt(topic: str) -> str:
    """
    Build a Weird AI prompt for reasoning through a parody-writing task.
    """

    return f"""You are Weird AI, a parody lyric assistant.

Topic: {topic}

Before writing the final parody lyric, plan your approach step-by-step:
1. Subject: What is the parody about? Which aspect of "{topic}" will be the focus?
2. Humour: What funny contrast, exaggeration, or pun will carry the joke? Aim for humour that fits "{topic}".
3. Rhyme: List various candidate rhymes that fit "{topic}" which could anchor the lyric.
4. Draft: Sketch a short structure for the chorus or verse, including metre or rhythm.

Upon completing these, write the final short parody lyric or chorus.
"""


print(build_parody_reasoning_prompt("networking cables and lost packets"))

You are Weird AI, a parody lyric assistant.

Topic: networking cables and lost packets

Before writing the final parody lyric, plan your approach step-by-step:
1. Subject: What is the parody about? Which aspect of "networking cables and lost packets" will be the focus?
2. Humour: What funny contrast, exaggeration, or pun will carry the joke? Aim for humour that fits "networking cables and lost packets".
3. Rhyme: List various candidate rhymes that fit "networking cables and lost packets" which could anchor the lyric.
4. Draft: Sketch a short structure for the chorus or verse, including metre or rhythm.

Upon completing these, write the final short parody lyric or chorus.



In [18]:
# Self-checks for build_parody_reasoning_prompt.

prompt = build_parody_reasoning_prompt("software testing")

assert isinstance(prompt, str), "The function should return a string."
assert "software testing" in prompt.lower(), "The topic should appear in the prompt."
assert "rhyme" in prompt.lower(), "The prompt should ask about rhyme words or rhyme planning."
assert "humor" in prompt.lower() or "funny" in prompt.lower(), "The prompt should include humor planning."
assert "final" in prompt.lower(), "The prompt should ask for a final lyric or chorus."

print("build_parody_reasoning_prompt self-checks passed.")

build_parody_reasoning_prompt self-checks passed.


## 6. When Reasoning Is Useful and When It Is Not

Reasoning is not always necessary.  A reasoning-style prompt may improve complex tasks, but it can also make simple tasks slower, longer, or more expensive.

For each task below, decide whether reasoning is useful.

In [14]:
tasks = [
    "Generate one rhyming word for 'code'.",
    "Write a four-line parody chorus about cybersecurity.",
    "Plan a parody verse that compares database normalization to organizing a messy closet.",
    "Translate the phrase 'hello world' into Spanish.",
    "Revise a parody lyric to make the joke clearer while keeping the rhyme scheme.",
]

for number, task in enumerate(tasks, start=1):
    print(f"{number}. {task}")

1. Generate one rhyming word for 'code'.
2. Write a four-line parody chorus about cybersecurity.
3. Plan a parody verse that compares database normalization to organizing a messy closet.
4. Translate the phrase 'hello world' into Spanish.
5. Revise a parody lyric to make the joke clearer while keeping the rhyme scheme.


### Exercise 4

For each task above, state whether reasoning would be useful.  Briefly explain why.

### Student Response: Exercise 4

TODO: Write your answer here.

## 7. Project Connection

In the previous lesson, Weird AI became an instruction-following parody assistant.  In this lesson, Weird AI begins moving toward a reasoning parody assistant.

The important shift is this:

```text
Instruction-following assistant:
"Do this task."

Reasoning assistant:
"Understand the task, plan the response, then do the task."
```

Later lessons can build on this by adding:

- Reasoning datasets
- Verifiers or reward functions
- Inference-time search or sampling
- Reinforcement learning
- Distillation from stronger reasoning examples

## Reflection Questions

Answer these in your assignment document.

1. What does reasoning mean in the context of LLMs?
2. What is chain-of-thought reasoning?
3. What is the difference between pattern matching and reasoning?
4. What is the difference between closed-world and open-world reasoning?
5. Why can a conventional LLM appear to reason even if it is primarily predicting likely text?
6. Why might reasoning be useful for Weird AI parody generation?
7. Give one example of a Weird AI task where reasoning would be useful.
8. Give one example of a Weird AI task where reasoning would probably be unnecessary.
9. Why can reasoning increase cost, latency, or verbosity?
10. What is one risk of trusting reasoning-looking text too much?

## Suggested Submission Items

Submit a Word document containing:

- Screenshots showing you completed the notebook cells
- Your answers to Exercises 1 through 4
- Your answers to the reflection questions
- A screenshot showing the self-check cells passed

Also commit your completed notebook and any assigned Python files to your GitHub repository.